# 03 · Modelo

Dos preguntas con el mismo método (`src/radar/modelo.py`):

1. **Radar**: con las descargas de usuario de los días 7, 14 o 21, ¿cuántas tendrá el paquete a los 30 días?
2. **Antes de lanzar**: sin ninguna descarga, ¿qué se puede decir?

Partición por fecha de lanzamiento (60 % entrenamiento, 20 % calibración, 20 % test), candidato elegido por validación cruzada temporal dentro del entrenamiento e intervalo del 90 % conformal con el bloque de calibración. El test se usa una sola vez.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

from radar import config
from radar import modelo as m

plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
ds = pd.read_csv(
    config.DIR_PROCESADO / "dataset_radar.csv", parse_dates=["fecha_lanzamiento"]
)
train, calib, test = m.particion_temporal(ds)
for nombre, parte in [("train", train), ("calibración", calib), ("test", test)]:
    print(
        f"{nombre:>12}: {len(parte):4d} paquetes, {parte.fecha_lanzamiento.min():%Y-%m-%d} → {parte.fecha_lanzamiento.max():%Y-%m-%d}"
    )

       train: 3130 paquetes, 2025-11-23 → 2026-04-28
 calibración: 1043 paquetes, 2026-04-28 → 2026-07-01
        test: 1044 paquetes, 2026-07-01 → 2026-10-09


## 1. Radar: comparación de candidatos

In [2]:
resultados = {
    d: m.entrenar(ds, m.variables_radar(d), columna_ratio=f"log_usuario_{d}d")
    for d in config.DIAS_LECTURA
}
cv = pd.concat([r.cv.assign(dia=d) for d, r in resultados.items()])
cv.pivot_table(
    index="modelo", columns="dia", values="mae_log", aggfunc=["mean", "std"]
).round(3)

mean                  std              
dia                   7      14     21     7      14     21
modelo                                                     
gradient_boosting  0.671  0.423  0.248  0.037  0.018  0.024
mediana            1.314  1.314  1.314  0.120  0.120  0.120
ratio_tipico       0.684  0.407  0.230  0.050  0.025  0.023
ridge              0.667  0.410  0.237  0.034  0.016  0.016

**Conclusión.** Con 3.130 lanzamientos de entrenamiento, los tres candidatos con información quedan muy juntos y muy lejos de la mediana (MAE de 0,67 frente a 1,31 a día 7). A día 7 Ridge baja algo el error del ratio típico (0,667 frente a 0,684), pero la diferencia queda dentro de la desviación entre pliegues. A días 14 y 21 gana la regla simple. Se respeta lo que dice la validación cruzada aunque las diferencias sean pequeñas.

## 2. Radar: resultado en test

In [3]:
filas = []
for d, r in resultados.items():
    filas.append({"día": d, "modelo": r.nombre, **r.test})
    for nombre, met in r.test_baselines.items():
        filas.append({"día": d, "modelo": nombre, **met})
pd.DataFrame(filas).set_index(["día", "modelo"]).round(3)

mae_log  mediana_error_log  factor_error_tipico  \
día modelo                                                               
7   ridge                0.648              0.589                1.802   
    mediana              1.294              0.974                2.649   
    ratio_tipico         0.624              0.530                1.699   
    gradient_boosting    0.646              0.564                1.758   
14  ratio_tipico         0.388              0.304                1.356   
    mediana              1.294              0.974                2.649   
    ridge                0.412              0.362                1.436   
    gradient_boosting    0.423              0.356                1.428   
21  ratio_tipico         0.218              0.149                1.161   
    mediana              1.294              0.974                2.649   
    ridge                0.244              0.203                1.225   
    gradient_boosting    0.250              0.197                1.218   

                       spearman  dentro_de_x2  cobertura_90  
día modelo                                                   
7   ridge                 0.849         0.604         0.920  
    mediana                 NaN         0.388           NaN  
    ratio_tipico          0.838         0.741           NaN  
    gradient_boosting     0.843         0.617           NaN  
14  ratio_tipico          0.924         0.895         0.917  
    mediana                 NaN         0.388           NaN  
    ridge                 0.928         0.910           NaN  
    gradient_boosting     0.927         0.900           NaN  
21  ratio_tipico          0.967         0.950         0.895  
    mediana                 NaN         0.388           NaN  
    ridge                 0.968         0.968           NaN  
    gradient_boosting     0.968         0.959           NaN

**Conclusión.** En los 1.044 lanzamientos más recientes, que ningún modelo vio, el radar se desvía en el paquete típico un factor ×1,80 a día 7, ×1,36 a día 14 y ×1,16 a día 21, y su rango del 90 % cubre entre el 89 % y el 92 % de los casos. A día 7, Ridge (×1,80) queda en test ligeramente por detrás del ratio típico (×1,70): la ventaja que tenía en la validación cruzada no se mantiene. Es la misma lección que en otros proyectos: con una relación casi lineal en log, la regla simple ya está cerca del techo.

## 3. ¿Dónde se equivoca?

In [4]:
r7 = resultados[7]
te = m.preparar_variables(test, r7.categorias)
pred = r7.modelo.predict(te[r7.variables.todas])
errores = test.assign(
    error=np.log1p(test["usuario_30d"]) - pred,
    tramo=pd.cut(
        test["usuario_7d"],
        [-1, 10, 30, 100, 300, np.inf],
        labels=["≤10", "11–30", "31–100", "101–300", ">300"],
    ),
)
errores.groupby("tramo", observed=True)["error"].agg(
    n="size", sesgo="median", error_tipico=lambda e: np.exp(e.abs().median())
).round(3)

,n,sesgo,error_tipico
tramo,,,
≤10,13,-0.207,1.487
11–30,304,-0.508,1.878
31–100,416,-0.377,1.833
101–300,201,-0.389,1.739
>300,110,-0.128,1.695


In [5]:
errores.groupby("categoria")["error"].agg(
    n="size", sesgo="median", error_tipico=lambda e: np.exp(e.abs().median())
).round(3)

,n,sesgo,error_tipico
categoria,,,
Agentes y LLM,537,-0.435,1.858
Datos y ciencia,145,-0.389,1.708
Herramientas de desarrollo,78,-0.165,1.763
Machine learning,118,-0.440,1.838
Otras,88,-0.235,1.776
Sistema y automatización,14,0.116,2.035
Web y APIs,64,-0.460,1.754


**Conclusión.** En el periodo de test el modelo sobreestima casi en todos los tramos: el sesgo mediano ronda −0,4 en log, es decir, lo real queda en torno a un 30 % por debajo de lo estimado. No es un problema de un tipo de paquete (pasa en todas las categorías), sino del tiempo: el multiplicador del día 7 al 30 era de ×2,0 a ×2,5 en los meses de entrenamiento y de ×1,6 a ×2,1 en los de test. El rango conformal lo absorbe en parte (cubre el 92 % de los casos), pero es lo primero que hay que vigilar al reentrenar cada mes.

## 4. ¿Qué pesa en la estimación?

In [6]:
tr7 = m.preparar_variables(train, r7.categorias)
ridge = m.candidatos(r7.variables, None)["ridge"].fit(
    tr7[r7.variables.todas], np.log1p(train["usuario_30d"])
)
imp = permutation_importance(
    ridge,
    te[r7.variables.todas],
    np.log1p(test["usuario_30d"]),
    n_repeats=10,
    random_state=m.SEMILLA,
    scoring="neg_mean_absolute_error",
)
pd.Series(imp.importances_mean, index=r7.variables.todas).sort_values(
    ascending=False
).round(3)

log_usuario_7d      1.110
log_total_7d        0.033
organizacion        0.025
subidas_7d          0.005
log_dependencias    0.005
categoria_modelo    0.003
dtype: float64

**Conclusión.** Casi todo el peso está en las descargas de usuario acumuladas: barajarlas dispara el error. Las totales y las subidas aportan algo (descuentan parte del ruido de publicar), y la categoría, el propietario o las dependencias casi nada, como anticipaba el EDA.

## 5. Antes de lanzar

In [7]:
previo = m.entrenar(ds, m.VARIABLES_PREVIAS)
print("Elegido por CV:", previo.nombre)
print("Test:", previo.test)
print("Mediana:", previo.test_baselines["mediana"])

Elegido por CV: gradient_boosting
Test: {'mae_log': 1.2523, 'mediana_error_log': 0.9119, 'factor_error_tipico': 2.489, 'spearman': 0.2195, 'dentro_de_x2': 0.4013, 'cobertura_90': 0.8755}
Mediana: {'mae_log': 1.2942, 'mediana_error_log': 0.9743, 'factor_error_tipico': 2.649, 'spearman': None, 'dentro_de_x2': 0.3879}


**Conclusión.** Antes del lanzamiento casi no hay señal. El candidato elegido (gradient boosting) mejora a la mediana por muy poco: error típico de ×2,49 frente a ×2,65 y un rango de unas 2,5 veces por arriba y por abajo, con una correlación de rangos de 0,22. Como mejora en el error medio y en el del paquete típico, la app enseña su cifra, pero siempre junto al rango de paquetes parecidos, que es lo que de verdad se puede decir.

## 6. Conclusiones y limitaciones

- **Lo que funciona:** con las descargas de usuario de la primera semana se estima el primer mes con un error típico de ×1,8 y un rango del 90 % que cubre el 92 % de los casos; a día 14 el error baja a ×1,36 y a día 21 a ×1,16.
- **Lo que no:** antes de publicar, la categoría, el propietario o las dependencias apenas dicen nada; y el veredicto «por debajo» casi nunca se puede afirmar antes del día 21, porque el cuartil inferior de la cohorte está pegado al suelo de bots que usan pip.
- **Limitaciones:** en el periodo de test el modelo sobreestima en torno a un 30 % porque el ritmo de crecimiento del primer mes ha bajado (apartado 3); las descargas de usuario cuentan instalaciones, no personas (un proyecto con CI instala cientos de veces); la muestra exige 10 estrellas el día de la descarga, lo que favorece a los proyectos más antiguos de la ventana; la categoría sale de reglas aproximadas.
- **Mantenimiento:** el workflow `actualizar-datos.yml` reentrena cada mes con los lanzamientos que van cerrando su primer mes y abre un Pull Request con las métricas para revisarlas antes de publicar.